<a href="https://colab.research.google.com/github/ishrat-waheed/learn_python/blob/main/cat_v_dog_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d sunilthite/cat-or-dog-image-classification

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/sunilthite/cat-or-dog-image-classification
License(s): other
100% 599M/599M [00:07<00:00, 83.5MB/s]



In [2]:
import zipfile
import os

# 'archive.zip' ki jagah apni file ka sahi naam likhen
zip_path = 'cat-or-dog-image-classification.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('my_dataset')

print("File unzip ho gayi hai! 'my_dataset' folder check karen.")

File unzip ho gayi hai! 'my_dataset' folder check karen.


In [3]:
import tensorflow as tf
from keras import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten

In [4]:
#generators
train_ds = tf.keras.utils.image_dataset_from_directory(
  directory = '/content/my_dataset/Train',
  labels = 'inferred',
  label_mode = 'int',
  batch_size = 32,
  image_size = (256, 256)
)

#generators
validation_ds = tf.keras.utils.image_dataset_from_directory(
  directory = '/content/my_dataset/Test',
  labels = 'inferred',
  label_mode = 'int',
  batch_size = 32,
  image_size = (256, 256)
)

Found 23650 files belonging to 2 classes.
Found 3863 files belonging to 2 classes.


In [5]:
#normalize
def process(image,label):
  image = tf.cast(image/255. ,tf.float32)
  return image,label

train_ds = train_ds.map(process)
validation_ds = validation_ds.map(process)

In [6]:
#create cnn model
model = Sequential()
model.add(Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))


model.add(Conv2D(128,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))


model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,847,297 (56.64 MB)

 Trainable params: 14,847,297 (56.64 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [8]:
history = model.fit(train_ds,epochs=10,validation_data=validation_ds)

Epoch 1/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 66s 76ms/step - accuracy: 0.6314 - loss: 0.6359 - val_accuracy: 0.7598 - val_loss: 0.5064
Epoch 2/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 53s 71ms/step - accuracy: 0.7747 - loss: 0.4707 - val_accuracy: 0.8115 - val_loss: 0.4311
Epoch 3/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 83s 73ms/step - accuracy: 0.8492 - loss: 0.3429 - val_accuracy: 0.8452 - val_loss: 0.4494
Epoch 4/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 57s 76ms/step - accuracy: 0.9209 - loss: 0.1924 - val_accuracy: 0.8651 - val_loss: 0.4429
Epoch 5/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 55s 74ms/step - accuracy: 0.9616 - loss: 0.1032 - val_accuracy: 0.8594 - val_loss: 0.5600
Epoch 6/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 56s 75ms/step - accuracy: 0.9703 - loss: 0.0830 - val_accuracy: 0.8977 - val_loss: 0.4315
Epoch 7/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 56s 76ms/step - accuracy: 0.9848 - loss: 0.0463 - val_accuracy: 0.8877 - val_loss: 0.5797
Epoch 8/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 55s 75ms/step - accuracy: 0.9904 - loss: 0.0273 - 